In [8]:
# Uninstall existing numpy and opencv
!pip uninstall -y numpy opencv-python-headless

# Install compatible versions
!pip install -q numpy==1.26.4 opencv-python-headless==4.9.0.80

Found existing installation: numpy 1.26.4
Uninstalling numpy-1.26.4:
  Successfully uninstalled numpy-1.26.4
Found existing installation: opencv-python-headless 4.9.0.80
Uninstalling opencv-python-headless-4.9.0.80:
  Successfully uninstalled opencv-python-headless-4.9.0.80
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
jax 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
thinc 8.3.6 requires numpy<3.0.0,>=2.0.0, but you have numpy 1.26.4 which is incompatible.
pytensor 2.35.1 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
jaxlib 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.


In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:


# ---------------------------
# Imports & Config
# ---------------------------
import os
import cv2
import time
import json
import math
import smtplib
import torch
import pandas as pd
from email.message import EmailMessage
from ultralytics import YOLO
from datetime import datetime, timedelta
from collections import defaultdict

# ---------------------------
# USER CONFIG - update paths & options here
# ---------------------------
BASE_DIR = "/content/drive/MyDrive/SafetyEye"
MODEL_PATH = f"{BASE_DIR}/models/best_model/best.pt"
VIDEO_DIR = f"{BASE_DIR}/realtime_inference/data"
OUTPUT_DIR = f"{BASE_DIR}/alert_system_outputs"
ALERT_LOG_CSV = os.path.join(OUTPUT_DIR, "alert_log.csv")
ANNOTATED_VIDEO_SUFFIX = "_alerted.mp4"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Inference config
CONF_THRESHOLD = 0.4
IOU_THRESHOLD = 0.45
IMGSZ = 640
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Rule config
# Map class name substrings to "helmet"/"vest" categories -- adapt to your class names
HELMET_KEYWORDS = ["helmet", "hardhat", "hard hat"]
VEST_KEYWORDS   = ["vest", "safety vest", "safety_vest"]

# Alert suppression: same person & same violation won't re-alert within this window
SUPPRESSION_SECONDS = 60

# Email alert config (disabled by default)
EMAIL_ENABLED = False
EMAIL_SMTP_SERVER = "smtp.example.com"
EMAIL_SMTP_PORT = 587
EMAIL_USERNAME = "you@example.com"
EMAIL_PASSWORD = "password"
EMAIL_FROM = "you@example.com"
EMAIL_TO = ["recipient@example.com"]

# Create output folders
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ---------------------------
# Helper functions
# ---------------------------
def now_iso():
    return datetime.utcnow().isoformat() + "Z"

def bbox_iou(boxA, boxB):
    # boxes are [x1,y1,x2,y2]
    xA = max(boxA[0], boxB[0]); yA = max(boxA[1], boxB[1])
    xB = min(boxA[2], boxB[2]); yB = min(boxA[3], boxB[3])
    interW = max(0, xB - xA); interH = max(0, yB - yA)
    interArea = interW * interH
    boxAArea = (boxA[2]-boxA[0])*(boxA[3]-boxA[1])
    boxBArea = (boxB[2]-boxB[0])*(boxB[3]-boxB[1])
    union = boxAArea + boxBArea - interArea
    return interArea / union if union > 0 else 0.0

def class_matches_keywords(label, keywords):
    lab = label.lower()
    return any(k in lab for k in keywords)

# ---------------------------
# Email helper (optional)
# ---------------------------
def send_email_alert(subject, body, smtp_conf):
    if not EMAIL_ENABLED:
        return False, "Email disabled"
    try:
        msg = EmailMessage()
        msg["From"] = smtp_conf["from"]
        msg["To"] = ", ".join(smtp_conf["to"])
        msg["Subject"] = subject
        msg.set_content(body)
        with smtplib.SMTP(smtp_conf["server"], smtp_conf["port"]) as s:
            s.starttls()
            s.login(smtp_conf["username"], smtp_conf["password"])
            s.send_message(msg)
        return True, "sent"
    except Exception as e:
        return False, str(e)

# ---------------------------
# Set up alert logging & suppression bookkeeping
# ---------------------------
# alert_records appended and flushed to ALERT_LOG_CSV
alert_records = []

# Keep track of last alert time for a unique key (video, person_bbox_center, violation_type)
last_alert_times = {}  # key -> datetime

def make_alert_key(video_name, person_bbox, violation):
    # Use rounded bbox center to reduce small jitter duplicates
    x1,y1,x2,y2 = person_bbox
    cx = int((x1+x2)/2); cy = int((y1+y2)/2)
    # round to 20-px grid to tolerate bbox jitter
    return f"{video_name}|{cx//20}_{cy//20}|{violation}"

def should_suppress(video_name, person_bbox, violation):
    key = make_alert_key(video_name, person_bbox, violation)
    now = datetime.utcnow()
    last = last_alert_times.get(key)
    if last and (now - last).total_seconds() < SUPPRESSION_SECONDS:
        return True
    last_alert_times[key] = now
    return False

def log_alert(record):
    alert_records.append(record)
    # flush periodically to CSV (append)
    df = pd.DataFrame([record])
    if not os.path.exists(ALERT_LOG_CSV):
        df.to_csv(ALERT_LOG_CSV, index=False)
    else:
        df.to_csv(ALERT_LOG_CSV, index=False, header=False, mode="a")

# ---------------------------
# Load model
# ---------------------------
print("Loading YOLO model:", MODEL_PATH, "on", DEVICE)
model = YOLO(MODEL_PATH)

# ---------------------------
# Process videos
# ---------------------------
video_files = [f for f in os.listdir(VIDEO_DIR) if f.lower().endswith((".mp4", ".avi", ".mov", ".mkv"))]
if not video_files:
    print("No video files found in:", VIDEO_DIR)
else:
    print(f"Found {len(video_files)} video(s). Starting processing...")

for vf in video_files:
    video_path = os.path.join(VIDEO_DIR, vf)
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print("❌ Could not open:", video_path)
        continue

    fps_in = cap.get(cv2.CAP_PROP_FPS) or 25.0
    w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    out_path = os.path.join(OUTPUT_DIR, vf.replace(".mp4", ANNOTATED_VIDEO_SUFFIX))
    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    writer = cv2.VideoWriter(out_path, fourcc, fps_in, (w, h))

    frame_idx = 0
    frame_times = []

    print("\n--- Processing", vf, "---")
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        t0 = time.time()
        # run inference (use predict or model(frame))
        results = model.predict(frame, conf=CONF_THRESHOLD, iou=IOU_THRESHOLD, imgsz=IMGSZ, device=DEVICE, verbose=False)
        res = results[0]

        # extract detections
        detections = []
        if hasattr(res, "boxes") and res.boxes is not None and len(res.boxes) > 0:
            boxes = res.boxes.xyxy.cpu().numpy()
            classes = res.boxes.cls.cpu().numpy()
            confs = res.boxes.conf.cpu().numpy()
            for box, cls, conf in zip(boxes, classes, confs):
                label = model.names[int(cls)]
                detections.append({
                    "bbox": [int(box[0]), int(box[1]), int(box[2]), int(box[3])],
                    "label": label,
                    "conf": float(conf)
                })

        # separate people & gear
        people = [d for d in detections if "person" in d["label"].lower()]
        gear = [d for d in detections if "person" not in d["label"].lower()]

        # For each person: find gear inside (simple containment / IoU)
        person_id = 0
        for person in people:
            person_id += 1
            px1,py1,px2,py2 = person["bbox"]
            # collect gear that intersects/in proximity with person bbox
            personal_gear = []
            for g in gear:
                gx1,gy1,gx2,gy2 = g["bbox"]
                iou = bbox_iou([px1,py1,px2,py2], [gx1,gy1,gx2,gy2])
                # choose containment or IoU > threshold (tweak if needed)
                if iou > 0.01:  # small overlap threshold; adjust
                    personal_gear.append(g)

            # Evaluate rules for this person
            has_helmet = any(class_matches_keywords(g["label"], HELMET_KEYWORDS) for g in personal_gear)
            has_vest   = any(class_matches_keywords(g["label"], VEST_KEYWORDS) for g in personal_gear)

            # You can extend to other PPE items in same manner

            violations = []
            if not has_helmet:
                violations.append("No Helmet")
            if not has_vest:
                violations.append("No Vest")

            # Build overlays: per-item color
            # Draw the person bbox and label per item
            # If item present -> green label, else red label
            # Draw person bbox with green if no violations else red
            person_color = (0,255,0) if not violations else (0,0,255)
            cv2.rectangle(frame, (px1,py1), (px2,py2), person_color, 2)

            # Compose status text: show present/missing for helmet & vest
            text_y = py1 - 10
            # Helmet status
            helmet_text = f"Helmet: {'YES' if has_helmet else 'NO'}"
            helmet_color = (0,255,0) if has_helmet else (0,0,255)
            cv2.putText(frame, helmet_text, (px1, text_y), cv2.FONT_HERSHEY_SIMPLEX, 0.6, helmet_color, 2)
            text_y -= 20
            # Vest status
            vest_text = f"Vest: {'YES' if has_vest else 'NO'}"
            vest_color = (0,255,0) if has_vest else (0,0,255)
            cv2.putText(frame, vest_text, (px1, text_y), cv2.FONT_HERSHEY_SIMPLEX, 0.6, vest_color, 2)
            text_y -= 20

            # Draw small boxes for gear detected (optional)
            for g in personal_gear:
                gx1,gy1,gx2,gy2 = g["bbox"]
                g_label = g["label"]
                g_conf = g["conf"]
                # gear color green
                cv2.rectangle(frame, (gx1,gy1), (gx2,gy2), (0,255,0), 1)
                cv2.putText(frame, f"{g_label} {g_conf:.2f}", (gx1, gy1-6), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0,255,0), 1)

            # If violation(s) -> create alert (subject to suppression)
            for v in violations:
                if should_suppress(vf, person["bbox"], v):
                    # suppressed, skip alerting
                    pass
                else:
                    # Compose alert record
                    timestamp = now_iso()
                    record = {
                        "timestamp": timestamp,
                        "video": vf,
                        "frame": int(frame_idx),
                        "person_id": f"{vf}_p{person_id}",
                        "violations": ";".join(violations),
                        "person_bbox": person["bbox"]
                    }
                    # Console alert
                    print(f"ALERT | {timestamp} | video={vf} frame={frame_idx} person={record['person_id']} violations={record['violations']}")
                    # Save to CSV
                    log_alert(record)
                    # Optional: send email
                    if EMAIL_ENABLED:
                        subject = f"[SafetyEye] Violation: {v} in {vf}"
                        body = f"Time: {timestamp}\nVideo: {vf}\nFrame: {frame_idx}\nPerson: {record['person_id']}\nViolations: {record['violations']}\nBBox: {record['person_bbox']}"
                        smtp_conf = {
                            "server": EMAIL_SMTP_SERVER,
                            "port": EMAIL_SMTP_PORT,
                            "username": EMAIL_USERNAME,
                            "password": EMAIL_PASSWORD,
                            "from": EMAIL_FROM,
                            "to": EMAIL_TO
                        }
                        ok, info = send_email_alert(subject, body, smtp_conf)
                        if not ok:
                            print("Email send failed:", info)

        # Draw global FPS
        t1 = time.time()
        frame_times.append(t1 - t0)
        fps = 1.0 / (t1 - t0) if (t1 - t0) > 0 else 0.0
        cv2.putText(frame, f"FPS: {fps:.2f}", (20,40), cv2.FONT_HERSHEY_SIMPLEX, 1.0, (255,255,255), 2)

        # write frame to output video
        writer.write(frame)
        frame_idx += 1

    cap.release()
    writer.release()
    # summary
    if frame_times:
        avg_fps = 1.0 / (sum(frame_times)/len(frame_times))
    else:
        avg_fps = 0.0
    print(f"Saved annotated -> {out_path} | Avg FPS: {avg_fps:.2f}")

# final: show location of CSV
print("\nAll done.")
print("Alert log CSV:", ALERT_LOG_CSV)
print("Annotated videos in:", OUTPUT_DIR)


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Loading YOLO model: /content/drive/MyDrive/SafetyEye/models/best_model/best.pt on cuda
Found 2 video(s). Starting processing...

--- Processing demo_1.mp4 ---


/tmp/ipython-input-4027941999.py:113: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  now = datetime.utcnow()
/tmp/ipython-input-4027941999.py:58: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().isoformat() + "Z"


ALERT | 2025-11-04T14:22:40.090345Z | video=demo_1.mp4 frame=0 person=demo_1.mp4_p1 violations=No Helmet
ALERT | 2025-11-04T14:22:40.773504Z | video=demo_1.mp4 frame=0 person=demo_1.mp4_p2 violations=No Helmet;No Vest
ALERT | 2025-11-04T14:22:40.779167Z | video=demo_1.mp4 frame=0 person=demo_1.mp4_p2 violations=No Helmet;No Vest
ALERT | 2025-11-04T14:22:40.801071Z | video=demo_1.mp4 frame=0 person=demo_1.mp4_p3 violations=No Helmet;No Vest
ALERT | 2025-11-04T14:22:40.806188Z | video=demo_1.mp4 frame=0 person=demo_1.mp4_p3 violations=No Helmet;No Vest
ALERT | 2025-11-04T14:22:40.857727Z | video=demo_1.mp4 frame=1 person=demo_1.mp4_p1 violations=No Helmet;No Vest
ALERT | 2025-11-04T14:22:40.862729Z | video=demo_1.mp4 frame=1 person=demo_1.mp4_p2 violations=No Helmet;No Vest
ALERT | 2025-11-04T14:22:40.867741Z | video=demo_1.mp4 frame=1 person=demo_1.mp4_p2 violations=No Helmet;No Vest
ALERT | 2025-11-04T14:22:40.873367Z | video=demo_1.mp4 frame=1 person=demo_1.mp4_p3 violations=No Helmet